# 05 - Integrated Pipeline: Clinical Triage Demo

Main integrative artifact for the Integrated AI System. This notebook ties together every component of the system and demonstrates three end-to-end runs through the agent orchestrator.

## Lineage map (prior capstone projects)

| Prior project | Concept adapted (code rebuilt) | Where in this system |
|---|---|---|
| **P2** - Data & Statistical Reasoning | Initial Data Analysis discipline, chi-square + Cramer's V, limitations/bias framing | `notebooks/01_data_exploration.ipynb` |
| **P3** - Machine Learning (K-Means RFM) | `log1p` + `StandardScaler` + scikit-learn Pipeline / ColumnTransformer | `src/preprocessing.py`, `src/ml_model.py` |
| **P4** - Deep Learning (CNN with dropout) | Fixed seed PyTorch training loop, dropout, per-slice disaggregated evaluation | `src/dl_model.py`, slice metrics in both ML and DL notebooks |
| **P5** - Generative AI (VAE) | Responsible-AI framing for generative outputs (under-claim capability, structural mitigations baked into the prompt, mandatory disclaimer) | `src/genai_explainer.py` |
| **P6** - Agentic AI (Research Brief Agent) | Plan -> Route -> Synthesise -> Evaluate -> Revise loop, Chroma + OpenAI embeddings, INSUFFICIENT EVIDENCE escape valve, refusal list, runtime caps, sha256 ingest manifest, JSONL run log | `src/rag/`, `src/agent_orchestrator.py`, `src/safeguards.py`, `outputs/run_log.jsonl` |

**Educational artifact only. Not for clinical use.**

## System architecture

```mermaid
flowchart TD
    ENTRY[Demo invocation<br/>run_all.py step 6 / notebook 05<br/>picks 3 demo patients + fixed clinician request<br/><i>run_all.py</i>] --> A
    A[Patient record<br/>UCI Heart Disease<br/><i>src/data_loader.py</i>] --> B[Preprocessing<br/>log1p + StandardScaler + one-hot<br/><i>src/preprocessing.py</i>]

    subgraph SCORING[Deterministic scoring]
        direction LR
        C[ML model<br/>HistGradientBoosting<br/><i>src/ml_model.py</i>]
        D[DL model<br/>PyTorch MLP<br/><i>src/dl_model.py</i>]
        E[Ensemble + tier + confidence flag<br/><i>src/decision.py</i>]
        C --> E
        D --> E
    end

    B --> C
    B --> D

    E --> F[Agent orchestrator<br/>plan / retrieve / explain / evaluate / revise<br/><i>src/agent_orchestrator.py</i>]

    subgraph INGEST[RAG ingestion - build time, idempotent]
        direction LR
        P[knowledge_base/*.md<br/>clinical guideline markdown]
        Q[Header-aware chunker<br/>split on H1/H2, max 1500 chars<br/><i>src/rag/knowledge_base.py</i>]
        R[OpenAI embeddings<br/>text-embedding-3-small<br/><i>src/rag/embeddings.py</i>]
        S[ChromaDB persistent<br/>HNSW + SQLite<br/><i>src/rag/vector_store.py</i>]
        T[sha256 manifest<br/>ingest_manifest.json<br/><i>src/rag/manifest.py</i>]
        P --> Q --> R --> S
        P -. hash check .-> T
        T -. skip unchanged .-> R
    end

    subgraph AGENTIC[Agentic loop - query time]
        direction LR
        G[RAG retriever<br/>top-k cosine over ChromaDB<br/><i>src/rag/retriever.py</i>]
        H[GenAI explainer<br/>OpenAI LLM + bracketed citations<br/><i>src/genai_explainer.py</i>]
        J[Evaluator-critic<br/>pass / score / issues<br/><i>src/agent_orchestrator.py</i>]
    end

    S -. read .-> G
    F --> G --> H
    H --> J
    J -. revise loop .-> H

    subgraph SAFETY[Safety side-channel]
        K[Refusal substring list + caps<br/><i>src/safeguards.py</i>]
    end
    F -. guardrail check .-> K

    H --> L[Clinician-facing explanation]
    K -. refusal .-> L

    subgraph AUDIT[Audit trail]
        direction LR
        M[outputs/run_log.jsonl<br/>append-only events<br/><i>src/agent_orchestrator.py</i>]
        N[docs/transcripts/<br/>per-run markdown<br/><i>src/transcripts.py</i>]
    end

    F -. log every event .-> M
    L -. persist .-> N
```

In [1]:
import sys, json
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.agent_orchestrator import run
from src.data_loader import load_heart_disease
from src.decision import load_default_scorers, score_cohort
from src.preprocessing import split_and_preprocess
from src.rag.retriever import ingest
from src.transcripts import save_transcript

TRANSCRIPTS = PROJECT_ROOT / 'docs' / 'transcripts'

## 1. Load data, models, and ingest KB

In [2]:
df = load_heart_disease()
split = split_and_preprocess(df)
ml, dl, pp = load_default_scorers(split)
ingest_summary = ingest()  # idempotent
print('ingest summary:', ingest_summary)

ingest summary: {'changed_files': [], 'chunks_added': 0, 'total_chunks': 29}


## 2. Score the entire test cohort
Lineage: P3 - 'report cohort sizes alongside the metric'. Here we report tier counts, model-agreement rate, and the resulting workload split a clinician would see.

In [3]:
cohort = score_cohort(split.X_test, ml, dl, pp)
cohort['agreement'] = (abs(cohort['ml_prob'] - cohort['dl_prob']) <= 0.20)
print('Tier counts:')
print(cohort['tier'].value_counts())
print()
print('Model-agreement rate:', round(cohort['agreement'].mean() * 100, 1), '%')
cohort.head(10)

Tier counts:
tier
high        29
low         28
moderate     4
Name: count, dtype: int64

Model-agreement rate: 73.8 %


,ml_prob,dl_prob,ensemble_prob,tier,agreement
219,0.564835,0.183387,0.374111,moderate,False
271,0.824733,0.778227,0.801480,high,True
89,0.001141,0.081391,0.041266,low,True
101,0.007796,0.088258,0.048027,low,True
67,0.060455,0.436182,0.248318,low,False
244,0.005871,0.117647,0.061759,low,True
185,0.216814,0.122087,0.169450,low,True
233,0.032227,0.508146,0.270186,low,False
168,0.990341,0.624691,0.807516,high,False
197,0.068564,0.394545,0.231554,low,False


## 3. Pick three demonstrative patients
- **Demo 1 - Happy path**: a high-tier, high-agreement positive case.
- **Demo 2 - Low-confidence borderline**: ml and dl disagree.
- **Demo 3 - Refusal**: a banned user request, regardless of patient.

In [4]:
cohort_with_y = cohort.join(split.y_test.rename('y_true'))
demo1 = cohort_with_y[(cohort_with_y['tier'] == 'high') & (cohort_with_y['agreement']) & (cohort_with_y['y_true'] == 1)]
demo1_idx = demo1.index[0] if len(demo1) else cohort_with_y[cohort_with_y['tier'] == 'high'].index[0]
demo2 = cohort_with_y[~cohort_with_y['agreement']].sort_values('ensemble_prob')
demo2_idx = demo2.index[len(demo2) // 2]  # middle borderline case
print('demo1 index:', demo1_idx, '|', cohort_with_y.loc[demo1_idx].to_dict())
print('demo2 index:', demo2_idx, '|', cohort_with_y.loc[demo2_idx].to_dict())

demo1 index: 137 | {'ml_prob': 0.998146586075733, 'dl_prob': 0.9302404522895813, 'ensemble_prob': 0.9641935191826572, 'tier': 'high', 'agreement': True, 'y_true': 1}
demo2 index: 243 | {'ml_prob': 0.9705871549306182, 'dl_prob': 0.6247329711914062, 'ensemble_prob': 0.7976600630610122, 'tier': 'high', 'agreement': False, 'y_true': 1}


## 4. Demo 1 - Happy path

In [5]:
features1 = split.X_test.loc[[demo1_idx]]
request1 = 'Summarise cardiovascular risk for this patient.'
result1 = run(request1, features1, ml, dl, pp)
transcript1 = save_transcript(result1, label='demo1_happy_path', out_dir=TRANSCRIPTS)
print('transcript:', transcript1.relative_to(PROJECT_ROOT))
print()
print(result1.explanation['text'])

transcript: docs\transcripts\demo1_happy_path_f8876174.md

High risk tier with ensemble probability of 0.964 (low<0.3, high>=0.7).  
Model agreement: high confidence.

- Age of 62 years, where cardiovascular disease prevalence rises with age, roughly doubling every decade after 55 [S3].
- Elevated cholesterol level (chol=281.0), indicating dyslipidemia, which is a major modifiable risk factor for cardiovascular events [S2].
- Presence of multi-vessel disease (ca=1.0), which substantially raises the probability of coronary disease [S1].
- ST depression on exercise (oldpeak=1.4), a classical positive stress test indicative of ischemia [S1].
- Reversible perfusion defect (thal=7.0), which most directly indicates inducible ischemia [S1].

What this system does not know: The model lacks information on smoking status, family history of cardiovascular disease, body mass index (BMI), HbA1c levels, LDL/HDL cholesterol ratios, current medications, and symptom acuity.

DISCLAIMER (include verbati

In [6]:
print('evaluator verdict:')
print(json.dumps(result1.evaluation, indent=2))
print('revised:', result1.revised)

evaluator verdict:
{
  "pass": true,
  "score": 10,
  "issues": [],
  "instructions": ""
}
revised: False


## 5. Demo 2 - Low-confidence borderline (model disagreement)

In [7]:
features2 = split.X_test.loc[[demo2_idx]]
request2 = 'Summarise cardiovascular risk for this patient.'
result2 = run(request2, features2, ml, dl, pp)
transcript2 = save_transcript(result2, label='demo2_low_confidence', out_dir=TRANSCRIPTS)
print('transcript:', transcript2.relative_to(PROJECT_ROOT))
print()
print(result2.explanation['text'])

transcript: docs\transcripts\demo2_low_confidence_0769064f.md

High risk tier with an ensemble probability of 0.798 (low<0.3, high>=0.7).  
Model agreement note: low confidence; human review recommended.

- **Typical angina (cp = 1)** indicates a high pretest probability of obstructive coronary disease [S1].
- **Elevated systolic blood pressure (trestbps = 134)** suggests hypertension, a major contributor to cardiovascular risk [S2].
- **Dyslipidemia indicated by cholesterol level (chol = 234)** aligns with the risk factor of elevated LDL cholesterol [S2].
- **ST depression (oldpeak = 2.6)** at peak exercise is a classical marker of inducible ischemia [S1].
- **Multi-vessel disease (ca = 2)** indicates a higher probability of coronary disease [S1].

What this system does not know: smoking status, family history of cardiovascular disease, body mass index (BMI), HbA1c levels, LDL/HDL cholesterol ratios, current medications, and symptom acuity.

Educational artifact only. Not for clinical

In [8]:
print('patient score:')
print(json.dumps(result2.score, indent=2))
print('evaluator verdict:')
print(json.dumps(result2.evaluation, indent=2))

patient score:
{
  "ml_prob": 0.9705871549306182,
  "dl_prob": 0.6247329711914062,
  "ensemble_prob": 0.7976600630610122,
  "tier": "high",
  "confidence": "low",
  "low_threshold": 0.3,
  "high_threshold": 0.7
}
evaluator verdict:
{
  "pass": false,
  "score": 7,
  "issues": [
    "Missing citations for some clinical claims",
    "Low-confidence case not properly surfaced"
  ],
  "instructions": "Add citations for all clinical claims made in the explanation. Explicitly state the uncertainty due to the low confidence in the model agreement note."
}


## 6. Demo 3 - Refusal
Lineage: P6 substring refusal list. No LLM call is made; the orchestrator refuses before any expensive step.

In [9]:
features3 = split.X_test.iloc[[0]]
request3 = 'Prescribe a medication and dosage for this patient.'
result3 = run(request3, features3, ml, dl, pp)
transcript3 = save_transcript(result3, label='demo3_refusal', out_dir=TRANSCRIPTS)
print('transcript:', transcript3.relative_to(PROJECT_ROOT))
print()
print('refused:', result3.refused, '-', result3.refusal_reason)

transcript: docs\transcripts\demo3_refusal_aabd2325.md

refused: True - matched refusal substring: prescribe


## 7. Audit the run log
Every step of every run is appended to `outputs/run_log.jsonl`. Below: the events from the last run only (most recent run_id).

In [10]:
log_path = PROJECT_ROOT / 'outputs' / 'run_log.jsonl'
if log_path.exists():
    lines = log_path.read_text(encoding='utf-8').splitlines()
    print(f'Total events logged across all runs: {len(lines)}')
    last_run_id = result3.run_id
    print(f'\nEvents for last run ({last_run_id}):')
    for raw in lines:
        rec = json.loads(raw)
        if rec.get('run_id') == last_run_id:
            print('-', rec['event'], {k: v for k, v in rec.items() if k not in ('event', 'ts', 'run_id')})
else:
    print('no run log yet')

Total events logged across all runs: 50

Events for last run (aabd2325):
- request {'user_request': 'Prescribe a medication and dosage for this patient.', 'n_features_rows': 1}
- refusal {'reason': 'prescribe'}


## 8. What this demo shows (for the synthesis paper)
- All five prior-project domains are exercised in a single run: tabular preprocessing (P3), ML score (P3), DL score (P4 discipline), RAG + evaluator-critic loop (P6), generative explanation with structural mitigations (P5), and statistical/IDA framing (P2) implicit in the limitations sections of every component.
- The system's three observable behaviours - confident explanation, low-confidence surfacing, and refusal - all come from explicit, auditable code, not from prompt-only safety. Each is logged.
- The same per-sex AUC gap that appeared in both ML and DL components is data-driven, not model-driven; it is acknowledged in the model card retrieved at explanation time.
- Persisted transcripts under `docs/transcripts/` are the verifiable evidence the mentor reviewer can audit.